# Contract Intelligence Pipeline — Combined Week 2 + Week 3

```
┌─────────────────────────────────────────────────────────────────────┐
│  SECTION 1 — PDF INGESTION                                          │
│  Read all .pdf files from a folder                                  │
│  Extract text  →  Clean  →  Segment into clauses                    │
│  NER (SpaCy)   →  Keyword classify  →  Per-doc JSON                 │
│                             corpus_clauses.json (merged)            │
└───────────────────────────────┬─────────────────────────────────────┘
                                │
┌───────────────────────────────▼─────────────────────────────────────┐
│  SECTION 2 — REGEX BASELINE + EVALUATION                            │
│  Load corpus_clauses.json  →  DataFrame                             │
│  Enhanced regex classifier (scored multi-pattern)                   │
│  Zero-shot NLI ground truth (local, no API key)                     │
│  Metrics: F1 / Precision / Recall / Accuracy / Confusion Matrix     │
└───────────────────────────────┬─────────────────────────────────────┘
                                │
┌───────────────────────────────▼─────────────────────────────────────┐
│  SECTION 3 — EMBEDDING + RETRIEVAL (RAG)                            │
│  Embed clauses with SentenceTransformer                             │
│  Test 3 embedding models on sample queries                          │
│  Evaluate retrieval accuracy (top-k hit rate)                       │
└─────────────────────────────────────────────────────────────────────┘
```

**JSON clause schema** (every clause across all PDFs):
```json
{
  "clause_id":       "affirm_merchant_agreement__42",
  "local_id":        42,
  "source_file":     "affirm_merchant_agreement.pdf",
  "clause_text":     "Merchant shall pay all applicable processing fees...",
  "category":        "Payment",
  "entities":        [{"text": "Affirm", "label": "ORG", ...}]
}
```

## 0. Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q pypdf spacy transformers scikit-learn \
                                     torch sentence-transformers \
                                     pandas matplotlib seaborn tqdm
!{sys.executable} -m spacy download en_core_web_sm -q
print('All dependencies installed.')

## 1. Configuration — Set Paths Here

In [ ]:
import os, re, json, time
from pathlib import Path
from collections import defaultdict

import spacy
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.metrics import (
    classification_report, f1_score, accuracy_score,
    precision_score, recall_score, confusion_matrix,
)
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer

# ── Mount Google Drive (Colab) ────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── ✏️  Edit these two paths ─────────────────────────────────────────────────
PDF_FOLDER = '/content/drive/MyDrive/Colab Notebooks/project'         # folder with your PDFs
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/project/outputs' # where outputs are saved
# ─────────────────────────────────────────────────────────────────────────────

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

pdf_files = sorted(Path(PDF_FOLDER).glob('*.pdf'))
print(f'Device     : {DEVICE}')
print(f'PDF folder : {PDF_FOLDER}')
print(f'Output dir : {OUTPUT_DIR}')
print(f'PDFs found : {len(pdf_files)}')
for p in pdf_files:
    print(f'  {p.name}')

In [ ]:
# ── DEMO: auto-generate synthetic PDFs when folder is empty ───────────────────
# Remove / skip this cell if you already have real PDFs in PDF_FOLDER.
if not pdf_files:
    print('No PDFs found. Creating demo fintech contracts...')
    try:
        from fpdf import FPDF
    except ImportError:
        !{sys.executable} -m pip install -q fpdf2
        from fpdf import FPDF

    DEMO = {
        'affirm_merchant_agreement.pdf': [
            'MERCHANT AGREEMENT — entered into between Affirm, Inc. and Merchant (Peloton Interactive Inc.).',
            '1. PAYMENT TERMS. Merchant shall pay all applicable processing fees per Schedule A. Settlement occurs within two (2) business days. BNPL MDR is 2.9% plus $0.30 per transaction. Late payments accrue interest at 1.5% per month. ACH wire transfer fees apply to disbursements.',
            '2. TERM AND TERMINATION. Either party may terminate upon ninety (90) days written notice. Affirm may terminate immediately for AML violations or fraudulent activity. Upon termination, all outstanding settlements shall be completed within the notice period.',
            '3. CONFIDENTIALITY. Each party shall maintain strict confidentiality of non-public financial data, transaction records, and pricing schedules. KYC documentation and AML screening results shall not be disclosed except as required by court order. All proprietary information protected.',
            '4. LIMITATION OF LIABILITY. In no event shall Affirm be liable for indirect or consequential damages. Aggregate liability shall not exceed fees paid in the preceding twelve months. Force majeure events limit liability of affected party.',
            '5. MERCHANT OBLIGATIONS. Merchant must implement PCI-DSS compliant environment and shall provide annual AOC certification within thirty (30) days after audit. Merchant is required to perform enhanced due diligence on all high-risk transactions and must report suspected fraud within 24 hours.',
            '6. GOVERNING LAW. This Agreement is governed by the laws of the State of California. Disputes shall be resolved by binding arbitration in San Francisco, California under JAMS rules.',
            '7. REPRESENTATIONS AND WARRANTIES. Each party warrants full authority to enter this Agreement. Affirm warrants its platform is PCI-DSS Level 1 certified throughout the term of the agreement.',
            '8. INTELLECTUAL PROPERTY. Affirm retains all rights in its platform, trademarks, and patents. Merchant receives a limited non-exclusive license to use the Affirm API solely for the purposes set forth herein.',
            '9. NOTICES. All notices must be delivered in writing via certified mail or email to the addresses set forth on the signature page. Written notice of any breach must be provided within five business days.',
            '10. ASSIGNMENT. Merchant may not assign any rights or obligations under this Agreement without prior written consent of Affirm.',
            '11. ENTIRE AGREEMENT. This Agreement supersedes all prior oral and written agreements and understandings between the parties relating to the subject matter hereof.',
        ],
        'lending_club_loan_agreement.pdf': [
            'LOAN AGREEMENT — LendingClub Bank, National Association, and Borrower.',
            '1. PAYMENT. Borrower shall repay the principal loan amount of $50,000 in 36 equal monthly installments. The annual percentage rate is 8.5%. All payments are due on the 15th of each month via ACH debit. Late payments incur a fee equal to 5% of the overdue amount.',
            '2. TERMINATION AND ACCELERATION. Lender may accelerate all outstanding obligations immediately upon Borrower insolvency, appointment of a receiver, or bankruptcy filing. Loan terminates upon full repayment of all principal, interest, and fees.',
            '3. CONFIDENTIALITY. LendingClub shall not disclose borrower financial data, credit information, or tax returns except as required by applicable law or regulatory directive. All banking secrecy obligations apply.',
            '4. LIMITATION OF LIABILITY. LendingClub disclaims all implied warranties regarding credit decisioning algorithms. Neither party shall be liable for losses caused by force majeure events including cyberattacks, regulatory actions, or acts of god.',
            '5. BORROWER OBLIGATIONS. Borrower shall maintain a minimum debt service coverage ratio (DSCR) of 1.25x at all times. Borrower must provide quarterly financial statements within 45 days of each quarter-end. Borrower is required to notify LendingClub of any material adverse change within 5 business days.',
            '6. REPRESENTATIONS AND WARRANTIES. Borrower represents that all financial statements provided are accurate and complete. Borrower warrants that no material litigation is pending that would affect its financial condition.',
            '7. GOVERNING LAW. This Agreement is governed by federal law and the laws of the State of Utah. Any disputes shall be submitted to the courts of Salt Lake City, Utah.',
            '8. ASSIGNMENT. Borrower may not assign any rights or obligations under this Agreement without prior written consent of LendingClub Bank.',
            '9. SEVERABILITY. If any provision of this Agreement is held invalid or unenforceable, the remaining provisions shall continue in full force and effect.',
            '10. AMENDMENTS. This Agreement may be amended only by a written instrument signed by authorized representatives of both parties.',
        ],
        'square_payment_services.pdf': [
            'PAYMENT SERVICES AGREEMENT — Square, Inc. and Merchant.',
            '1. FEES AND SETTLEMENT. Square charges a flat rate of 2.6% plus $0.10 per card-present transaction and 2.9% plus $0.30 per card-not-present transaction. Funds are settled to the Merchant linked bank account within one to two business days. Foreign currency conversion fees of 1.5% apply to international transactions.',
            '2. TERM AND TERMINATION. Square may suspend or terminate merchant accounts immediately upon evidence of excessive chargeback ratios exceeding 1% of monthly transaction volume. Either party may terminate this Agreement for convenience with 30 days written notice.',
            '3. CONFIDENTIALITY AND DATA SECURITY. Cardholder data, PAN information, and transaction records are Confidential Information subject to PCI-DSS. Square shall not sell or disclose merchant financial data to third parties without written consent. All KYC and AML data protected.',
            '4. LIMITATION OF LIABILITY. Square maximum aggregate liability is capped at fees paid in the prior 12 months. Square is not liable for consequential damages, lost profits, loss of business, or interruption losses arising from use of services.',
            '5. MERCHANT OBLIGATIONS. Merchant shall not process prohibited business types listed in the Acceptable Use Policy. Merchant must ensure all hardware complies with EMV chip standards. Merchant shall report suspected fraud or suspicious transactions within 24 hours of discovery.',
            '6. INTELLECTUAL PROPERTY. The Square mark, logo, and point-of-sale software are proprietary to Square. Merchant receives a revocable, non-exclusive, non-transferable license to use Square trademarks solely in connection with the services.',
            '7. INDEMNIFICATION. Merchant shall indemnify, defend, and hold harmless Square from any third-party claims, losses, or damages arising from Merchant violation of Card Network Rules or applicable law.',
            '8. AMENDMENTS. Square may modify these terms with 30 days prior written notice. Continued use of services after the effective date of the amendment constitutes acceptance of the amended terms.',
            '9. GOVERNING LAW. This Agreement is governed by the laws of the State of California, without regard to conflict of law principles. Disputes resolved by arbitration in San Francisco.',
            '10. ENTIRE AGREEMENT. This Agreement and its exhibits constitute the entire agreement between Square and Merchant and supersede all prior negotiations and oral agreements.',
        ],
    }

    Path(PDF_FOLDER).mkdir(parents=True, exist_ok=True)
    for fname, paras in DEMO.items():
        pdf = FPDF(); pdf.add_page(); pdf.set_font('Helvetica', size=10)
        for p in paras:
            pdf.multi_cell(0, 6, p); pdf.ln(3)
        pdf.output(str(Path(PDF_FOLDER) / fname))
        print(f'  Created: {fname}')

    pdf_files = sorted(Path(PDF_FOLDER).glob('*.pdf'))
    print(f'Demo corpus ready: {len(pdf_files)} PDFs')

---
# SECTION 1 — PDF Ingestion & Clause Extraction
*(All functions carried forward from Week 2 — unchanged)*

In [ ]:
# ── Week 2: clean_text ────────────────────────────────────────────────────────
def clean_text(text):
    """Remove encoding artifacts and normalise whitespace (Week 2, unchanged)."""
    cleaned = re.sub(r'\uf0a0', ' ', text)    # form-feed
    cleaned = re.sub(r'\u00a0', ' ', cleaned)  # non-breaking space
    cleaned = re.sub(r'\s+',    ' ', cleaned)  # collapse whitespace
    return cleaned.strip()

# ── Week 2: segment_into_clauses ─────────────────────────────────────────────
def segment_into_clauses(text):
    """Split cleaned text into clause-level segments (Week 2, unchanged)."""
    clause_delimiters = r'(?<=[.!?]) +|\n+|; +'
    clauses = re.split(clause_delimiters, text)
    return [c.strip() for c in clauses if c.strip()]

# ── Week 2: clause_categories ────────────────────────────────────────────────
clause_categories = {
    'Parties':                       ['merchant','company','party','agreement','affiliate','customer','supplier','provider','user','seller','buyer'],
    'Term and Termination':          ['term','terminate','termination','expiration','renewal','effective date','commencement date','end date','duration','period'],
    'Confidentiality':               ['confidential','disclosure','secret','proprietary','non-disclosure','nondisclosure','confidential information'],
    'Governing Law':                 ['governing law','jurisdiction','applicable law','venue','dispute resolution','arbitration','court','state','country'],
    'Payment':                       ['payment','fee','charge','invoice','remuneration','compensation','price','currency','due date','tax','late payment'],
    'Indemnification':               ['indemnify','indemnification','hold harmless','damages','losses','claims','liabilities'],
    'Limitation of Liability':       ['limit liability','limitation of liability','maximum liability','exclude liability','indirect damages','consequential damages'],
    'Representations and Warranties':['represent','warrant','representation','warranty','guarantee','covenant','accuracy','truthfulness'],
    'Intellectual Property':         ['intellectual property','patent','trademark','copyright','license','licence','infringement','ownership','rights'],
    'Force Majeure':                 ['force majeure','act of god','unforeseeable','circumstances','event beyond control','delay','failure to perform'],
    'Notices':                       ['notice','address','writing','mail','email','fax','delivery'],
    'Assignment':                    ['assign','assignment','transfer','delegate'],
    'Severability':                  ['severable','invalid','unenforceable','void','provision','clause'],
    'Amendments':                    ['amend','modify','change','alteration','in writing'],
    'Entire Agreement':              ['entire agreement','supersedes','prior agreements','oral agreements','understanding'],
    'General Provisions':            [],
}

# ── Week 2: classify_clause ───────────────────────────────────────────────────
def classify_clause(clause_text, categories):
    """Keyword-count classifier (Week 2, unchanged)."""
    lower = clause_text.lower()
    best, max_matches = 'General Provisions', 0
    for cat, kws in categories.items():
        if cat == 'General Provisions': continue
        m = sum(1 for kw in kws if kw in lower)
        if m > max_matches:
            max_matches, best = m, cat
    return best

# ── Week 2: extract_metadata ──────────────────────────────────────────────────
def extract_metadata(combined_legal_data, source_file):
    """Auto-extract document-level metadata from classified clauses."""
    orgs = set()
    dates = []
    for item in combined_legal_data:
        for ent in item.get('entities', []):
            if ent['label'] == 'ORG':
                orgs.add(ent['text'].strip())
            if ent['label'] == 'DATE' and len(dates) < 3:
                dates.append(ent['text'].strip())
    return {
        'source_file':    source_file,
        'document_type':  'Financial Services Agreement',
        'parties_found':  sorted(orgs)[:5],
        'dates_found':    dates,
        'total_clauses':  len(combined_legal_data),
    }

print('Week 2 functions loaded: clean_text, segment_into_clauses, classify_clause, extract_metadata')

In [ ]:
from pypdf import PdfReader

print('Loading SpaCy en_core_web_sm...')
nlp = spacy.load('en_core_web_sm')

def extract_text_from_pdf(path):
    """Extract raw text from all pages of a PDF (Week 2 approach)."""
    raw = ''
    with open(path, 'rb') as f:
        for page in PdfReader(f).pages:
            t = page.extract_text()
            if t: raw += t + '\n'
    return raw

def run_ner(clauses):
    """Run SpaCy NER on a list of clause strings (Week 2 approach)."""
    results = []
    for clause in clauses:
        doc = nlp(clause)
        results.append({
            'clause_text': clause,
            'entities': [
                {'text': e.text, 'label': e.label_,
                 'start_char': e.start_char, 'end_char': e.end_char}
                for e in doc.ents
            ]
        })
    return results

def process_single_pdf(path):
    """
    Full pipeline for one PDF — mirrors Week 2 flow.
    Returns a dict with metadata + clauses list.
    Each clause has a globally-unique clause_id.
    """
    stem  = path.stem                          # filename without extension
    raw   = extract_text_from_pdf(path)
    clean = clean_text(raw)
    segs  = segment_into_clauses(clean)
    ner   = run_ner(segs)

    clauses = []
    for i, seg in enumerate(segs):
        clauses.append({
            # ── globally unique id: filename__localindex ──────────────────────
            'clause_id':   f'{stem}__{i:04d}',
            'local_id':    i,
            'source_file': path.name,
            'clause_text': seg,
            'category':    classify_clause(seg, clause_categories),
            'entities':    ner[i]['entities'],
        })

    metadata = extract_metadata(clauses, path.name)

    return {
        'metadata': metadata,
        'clauses':  clauses,
    }

print('PDF processor ready.')

In [ ]:
# ── Process every PDF in the folder ──────────────────────────────────────────
corpus_docs = []
doc_stats   = []

print(f'Processing {len(pdf_files)} PDF(s)...\n')

for path in tqdm(pdf_files, desc='PDFs'):
    t0  = time.time()
    doc = process_single_pdf(path)
    elapsed = round(time.time() - t0, 2)

    corpus_docs.append(doc)

    # category counts for stats
    cat_counts = defaultdict(int)
    for c in doc['clauses']:
        cat_counts[c['category']] += 1

    doc_stats.append({
        'file':     path.name,
        'clauses':  len(doc['clauses']),
        'seconds':  elapsed,
        **dict(cat_counts)
    })

    # save per-document JSON (same shape as Week 2 output)
    per_doc_path = Path(OUTPUT_DIR) / f'{path.stem}_parsed.json'
    json.dump(doc, open(per_doc_path, 'w'), indent=4)
    print(f'  {path.name:50s}  {len(doc["clauses"]):4d} clauses  {elapsed}s  ->  {per_doc_path.name}')

# ── Save merged corpus JSON ───────────────────────────────────────────────────
corpus_path = Path(OUTPUT_DIR) / 'corpus_clauses.json'
json.dump(corpus_docs, open(corpus_path, 'w'), indent=4)
total_clauses = sum(len(d['clauses']) for d in corpus_docs)
print(f'\nCorpus saved → {corpus_path}')
print(f'Total clauses across all PDFs: {total_clauses}')

In [ ]:
# ── Per-document summary ─────────────────────────────────────────────────────
stats_df = pd.DataFrame(doc_stats)
print('=== Per-Document Summary ===')
print(stats_df[['file','clauses','seconds']].to_string(index=False))

# Corpus-level category distribution
all_clauses_flat = [c for d in corpus_docs for c in d['clauses']]
print(f'\n=== Corpus Category Distribution (total: {len(all_clauses_flat)}) ===')
print(pd.Series([c['category'] for c in all_clauses_flat]).value_counts().to_string())

In [ ]:
# ── Preview clause schema ────────────────────────────────────────────────────
# Show the structured JSON for the first 5 clauses of the first document
sample_doc = corpus_docs[0]
print(f'\nDocument: {sample_doc["metadata"]["source_file"]}')
print(f'Metadata: {json.dumps(sample_doc["metadata"], indent=2)}')
print(f'\nFirst 5 clauses:')
for c in sample_doc['clauses'][:5]:
    print()
    print(json.dumps({
        'clause_id':   c['clause_id'],
        'category':    c['category'],
        'clause_text': c['clause_text'][:120] + '...',
        'entities':    c['entities'][:2],
    }, indent=2))

In [ ]:
# ── Flat table view (Week 2 style display) ───────────────────────────────────
table_rows = []
for doc in corpus_docs:
    for c in doc['clauses']:
        if c['entities']:
            for ent in c['entities']:
                table_rows.append({
                    'clause_id':    c['clause_id'],
                    'source_file':  c['source_file'],
                    'category':     c['category'],
                    'clause_text':  c['clause_text'][:120],
                    'entity':       ent['text'],
                    'entity_type':  ent['label'],
                })
        else:
            table_rows.append({
                'clause_id':    c['clause_id'],
                'source_file':  c['source_file'],
                'category':     c['category'],
                'clause_text':  c['clause_text'][:120],
                'entity':       '',
                'entity_type':  '',
            })

display_df = pd.DataFrame(table_rows)
print(f'Flat table: {len(display_df)} rows  ({len(all_clauses_flat)} unique clauses)')
display(display_df.head(30))

---
# SECTION 2 — Regex Baseline + Evaluation

In [ ]:
# ── Load corpus JSON → flat DataFrame for evaluation ─────────────────────────
with open(corpus_path) as f:
    loaded_corpus = json.load(f)

rows = []
for doc in loaded_corpus:
    for c in doc['clauses']:
        rows.append({
            'clause_id':      c['clause_id'],
            'local_id':       c['local_id'],
            'source_file':    c['source_file'],
            'clause_text':    c['clause_text'],
            'week2_category': c['category'],   # label from Week 2 keyword classifier
        })

df = pd.DataFrame(rows)
df = df[df['clause_text'].str.len() >= 40].reset_index(drop=True)
print(f'Evaluation DataFrame: {len(df)} clauses from {df["source_file"].nunique()} file(s)')
print(f'\nWeek 2 category distribution:')
print(df['week2_category'].value_counts())

In [ ]:
# ── Enhanced regex baseline (same 16 categories, scored multi-pattern) ────────
#
# Improvement over Week 2 classify_clause:
#   Week 2: plain substring match, first-hit wins
#   Week 3: re.findall with \b boundaries, highest-score wins

REGEX_PATTERNS = {
    'Parties':                       r'(\bpart(?:y|ies)\b|\bmerchant\b|\bsupplier\b|\bprovider\b|\bcustomer\b|\bseller\b|\bbuyer\b|\baffiliate\b|\bhereinafter\b|\bentered\s+into\b)',
    'Term and Termination':          r'(\bterminat[ei]\w*\b|\bterm\s+of\b|\bexpir\w+\b|\brenewal\b|\bcancel\w*\b|\beffective\s+date\b|\bwithout\s+cause\b|\bfor\s+cause\b|\bwind\s*down\b|\baccelerat\w+\b|\bsuspend\w*\b)',
    'Confidentiality':               r'(\bconfidential\w*\b|\bnon.?disclosure\b|\bproprietary\b|\btrade\s+secret\b|\bNDA\b|\bprivacy\b|\bdisclose\b|\bsecrecy\b|\bKYC\b|\bAML\b|\bSAR\b)',
    'Governing Law':                 r'(\bgoverning\s+law\b|\bjurisdiction\b|\bapplicable\s+law\b|\bvenue\b|\bdispute\s+resolution\b|\barbitration\b|\bgoverned\s+by\b|\bstate\s+of\s+\w+\b|\bfederal\s+courts?\b)',
    'Payment':                       r'(\bpayment\b|\bfee[s]?\b|\binvoice\b|\bdisbursement\b|\bdue\s+date\b|\bwire\s+transfer\b|\bACH\b|\bprocessing\s+fee\b|\bsettle[sd]?\b|\bMDR\b|\binterchange\b|\bchargeback\b|\bcurrency\b)',
    'Indemnification':               r'(\bindemnif\w+\b|\bindemnit\w+\b|\bhold\s+harmless\b|\bthird.?party\s+claim\b|\blosses\b|\bfines\b|\bpenalt\w+\b)',
    'Limitation of Liability':       r'(\blimit\w*\s+liabilit\w+\b|\bliabilit\w+\s+cap\b|\bindirect\s+damages\b|\bconsequential\s+damages\b|\bpunitive\b|\bin\s+no\s+event\s+shall\b|\bnot\s+exceed\b|\bdisclaim\w*\b)',
    'Representations and Warranties':r'(\brepresent\w*\b|\bwarrant\w*\b|\bwarranties\b|\bguarantee[sd]?\b|\bcovenant[s]?\b|\baccuracy\b|\bhereby\s+represent\b)',
    'Intellectual Property':         r'(\bintellectual\s+property\b|\bpatent[s]?\b|\btrademark[s]?\b|\bcopyright[s]?\b|\blicen[sc]e[sd]?\b|\binfringement\b|\bownership\s+of\b|\broyalt\w+\b)',
    'Force Majeure':                 r'(\bforce\s+majeure\b|\bact\s+of\s+god\b|\bunforeseeable\b|\bbeyond\s+.{0,20}control\b|\bfailure\s+to\s+perform\b|\bpandemic\b)',
    'Notices':                       r'(\bnotice[s]?\b|\bwritten\s+notice\b|\bnotif\w+\b|\baddressed\s+to\b|\bcertified\s+mail\b|\bemail\s+to\b)',
    'Assignment':                    r'(\bassign\w*\b|\btransfer\w*\b|\bdelegate[sd]?\b|\bsuccessor[s]?\b|\bnovation\b)',
    'Severability':                  r'(\bseverab\w+\b|\bsevered\b|\binvalid\b|\bunenforceable\b|\bvoid\b|\bremaining\s+provision\b)',
    'Amendments':                    r'(\bamend\w*\b|\bmodif\w+\b|\balteration\b|\bsupplemental\b|\baddendum\b|\bwritten\s+consent\b)',
    'Entire Agreement':              r'(\bentire\s+agreement\b|\bsupersede[sd]?\b|\bprior\s+agreement\b|\boral\s+agreement\b)',
    'General Provisions':            r'(\bgeneral\s+provision\b|\bmiscellaneous\b)',
}

def regex_baseline_classify(text):
    """Score every category by regex hit count; return the highest scorer."""
    scores = {lbl: len(re.findall(pat, text, re.IGNORECASE))
              for lbl, pat in REGEX_PATTERNS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'General Provisions'

# Smoke test
smoke = [
    ('Merchant shall pay all processing fees within 30 days of invoice.',        'Payment'),
    ('Either party may terminate upon 90 days written notice for cause.',        'Term and Termination'),
    ('In no event shall Affirm be liable for consequential indirect damages.',   'Limitation of Liability'),
    ('All proprietary information and KYC data shall remain confidential.',      'Confidentiality'),
    ('Agreement governed by the laws of the State of California, arbitration.',  'Governing Law'),
    ('This Agreement supersedes all prior oral and written understandings.',      'Entire Agreement'),
]
print(f'{"Clause":55s}  {"Expected":30s}  {"Got":30s}  OK?')
print('-'*125)
for txt, exp in smoke:
    pred = regex_baseline_classify(txt)
    print(f'{txt[:54]:55s}  {exp:30s}  {pred:30s}  {"✓" if pred==exp else "✗"}')

In [ ]:
# Apply regex baseline to entire corpus
df['regex_label'] = df['clause_text'].apply(regex_baseline_classify)
print('=== Regex Baseline Label Distribution ===')
print(df['regex_label'].value_counts())
agree = (df['regex_label'] == df['week2_category']).sum()
print(f'\nAgreement with Week-2 keyword classifier: {agree}/{len(df)} ({100*agree/len(df):.1f}%)')

In [ ]:
# ── Zero-shot NLI ground-truth labels (local, no API key) ────────────────────
NLI_MODEL = 'cross-encoder/nli-MiniLM2-L6-H768'   # ~90 MB; swap for facebook/bart-large-mnli for better accuracy
print(f'Loading NLI labeler: {NLI_MODEL}  (Device: {DEVICE})')

nli = hf_pipeline('zero-shot-classification', model=NLI_MODEL,
                  device=0 if DEVICE=='cuda' else -1)

HYPOTHESES = {
    'Parties':                       'This clause identifies the parties or entities involved in the agreement.',
    'Term and Termination':          'This clause covers the duration, termination, cancellation or expiry of the agreement.',
    'Confidentiality':               'This clause covers confidentiality, non-disclosure, or protection of proprietary information.',
    'Governing Law':                 'This clause specifies the governing law, jurisdiction, or dispute resolution mechanism.',
    'Payment':                       'This clause is about payment, fees, invoicing, or financial settlement terms.',
    'Indemnification':               'This clause covers indemnification, hold harmless obligations, or third-party claims.',
    'Limitation of Liability':       'This clause limits liability of a party or excludes consequential damages.',
    'Representations and Warranties':'This clause contains representations, warranties, or guarantees made by a party.',
    'Intellectual Property':         'This clause covers intellectual property rights, licenses, patents, or trademarks.',
    'Force Majeure':                 'This clause addresses force majeure or events beyond the control of the parties.',
    'Notices':                       'This clause specifies how notices or communications must be delivered between parties.',
    'Assignment':                    'This clause governs the assignment or transfer of rights and obligations.',
    'Severability':                  'This clause addresses the severability of invalid or unenforceable provisions.',
    'Amendments':                    'This clause governs how the agreement may be amended or modified.',
    'Entire Agreement':              'This clause states that this document is the entire agreement between the parties.',
    'General Provisions':            'This clause contains general or miscellaneous boilerplate contract provisions.',
}
CANDIDATES = list(HYPOTHESES.values())
HYP2LABEL  = {v: k for k, v in HYPOTHESES.items()}

def nli_classify(text):
    r = nli(text[:512], candidate_labels=CANDIDATES, multi_label=False)
    return HYP2LABEL.get(r['labels'][0], 'General Provisions')

print('NLI labeler ready.')

In [ ]:
GT_CACHE = Path(OUTPUT_DIR) / 'ground_truth_labels.json'

if GT_CACHE.exists():
    print('Loading cached ground-truth labels...')
    gt_map = json.load(open(GT_CACHE))
    df['ground_truth'] = df['clause_text'].map(gt_map).fillna('General Provisions')
else:
    print(f'Labeling {len(df)} clauses with local NLI model...')
    gt_map = {}
    for text in tqdm(df['clause_text'], desc='NLI labeling'):
        gt_map[text] = nli_classify(text)
    df['ground_truth'] = df['clause_text'].map(gt_map)
    json.dump(gt_map, open(GT_CACHE, 'w'), indent=2)
    print(f'Cached → {GT_CACHE}')

print('\nGround-truth distribution:')
print(df['ground_truth'].value_counts())

In [ ]:
# ── Full metric suite ─────────────────────────────────────────────────────────
y_true = df['ground_truth'].tolist()
y_pred = df['regex_label'].tolist()
LABELS = sorted(set(y_true) | set(y_pred))

metrics = {
    'Accuracy':        round(accuracy_score(y_true, y_pred), 4),
    'F1_Macro':        round(f1_score(y_true, y_pred, average='macro',    labels=LABELS, zero_division=0), 4),
    'F1_Weighted':     round(f1_score(y_true, y_pred, average='weighted', labels=LABELS, zero_division=0), 4),
    'F1_Micro':        round(f1_score(y_true, y_pred, average='micro',    labels=LABELS, zero_division=0), 4),
    'Precision_Macro': round(precision_score(y_true, y_pred, average='macro', labels=LABELS, zero_division=0), 4),
    'Recall_Macro':    round(recall_score(y_true, y_pred, average='macro',    labels=LABELS, zero_division=0), 4),
}

print('='*52)
print('  REGEX BASELINE — AGGREGATE METRICS')
print('='*52)
for k, v in metrics.items():
    print(f'  {k:22s}: {v:.4f}')
print('='*52)

print('\n=== Per-Class Classification Report ===')
print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0, digits=4))

rpt = classification_report(y_true, y_pred, labels=LABELS, zero_division=0, output_dict=True)
per_class_df = pd.DataFrame({
    lbl: {'Precision': round(rpt[lbl]['precision'],4),
          'Recall':    round(rpt[lbl]['recall'],   4),
          'F1':        round(rpt[lbl]['f1-score'], 4),
          'Support':   int(rpt[lbl]['support'])}
    for lbl in LABELS if lbl in rpt
}).T.sort_values('F1', ascending=False)

print('\n=== Per-Class Table (sorted by F1) ===')
display(per_class_df)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Regex Baseline Evaluation', fontsize=15, fontweight='bold')

# 1 — Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=LABELS)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS,
            yticklabels=LABELS, ax=axes[0,0], linewidths=0.4)
axes[0,0].set_title('Confusion Matrix', fontweight='bold')
axes[0,0].set_xlabel('Predicted'); axes[0,0].set_ylabel('Actual')
axes[0,0].tick_params(axis='x', rotation=40, labelsize=8)
axes[0,0].tick_params(axis='y', rotation=0,  labelsize=8)

# 2 — Per-class P/R/F1 bar chart
ls = per_class_df.index.tolist()
x, w = np.arange(len(ls)), 0.26
axes[0,1].bar(x-w, per_class_df['Precision'], w, label='Precision', color='#1f4e79', alpha=0.85)
axes[0,1].bar(x,   per_class_df['Recall'],    w, label='Recall',    color='#2e75b6', alpha=0.85)
axes[0,1].bar(x+w, per_class_df['F1'],        w, label='F1',        color='#70ad47', alpha=0.85)
axes[0,1].set_xticks(x); axes[0,1].set_xticklabels(ls, rotation=40, ha='right', fontsize=8)
axes[0,1].set_ylim(0, 1.15); axes[0,1].set_title('Per-Class P / R / F1', fontweight='bold')
axes[0,1].legend(); axes[0,1].grid(axis='y', alpha=0.3)
for xi, f1 in zip(x, per_class_df['F1']):
    axes[0,1].text(xi+w, f1+0.02, f'{f1:.2f}', ha='center', fontsize=7, color='#375623')

# 3 — Aggregate metrics
bars = axes[1,0].barh(list(metrics.keys()), list(metrics.values()),
                      color=['#1f4e79','#2e75b6','#375623','#70ad47','#c55a11','#833c00'],
                      alpha=0.85, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, metrics.values()):
    axes[1,0].text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                   f'{val:.4f}', va='center', fontsize=10, fontweight='bold')
axes[1,0].set_xlim(0, 1.15); axes[1,0].set_title('Aggregate Metrics', fontweight='bold')
axes[1,0].grid(axis='x', alpha=0.3)

# 4 — Ground-truth vs regex distribution
all_cats = sorted(set(df['ground_truth']) | set(df['regex_label']))
gt_c = df['ground_truth'].value_counts(); rx_c = df['regex_label'].value_counts()
xd, wd = np.arange(len(all_cats)), 0.40
axes[1,1].bar(xd-wd/2, [gt_c.get(c,0) for c in all_cats], wd,
              label='Ground Truth (NLI)', color='#1f4e79', alpha=0.85)
axes[1,1].bar(xd+wd/2, [rx_c.get(c,0) for c in all_cats], wd,
              label='Regex Baseline', color='#ed7d31', alpha=0.85)
axes[1,1].set_xticks(xd); axes[1,1].set_xticklabels(all_cats, rotation=40, ha='right', fontsize=8)
axes[1,1].set_title('Label Distribution: GT vs Regex', fontweight='bold')
axes[1,1].legend(); axes[1,1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR)/'baseline_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved baseline_evaluation.png')

---
# SECTION 3 — Embedding Generation & Retrieval (RAG)

Each clause from the corpus JSON becomes a searchable vector unit.
Three embedding models are benchmarked on sample fintech queries.

In [ ]:
# ── Prepare clause list from corpus (all PDFs, all clauses) ──────────────────
# Each entry keeps clause_id so retrieved clauses are traceable back to source.
clause_records = [
    {'clause_id':   c['clause_id'],
     'source_file': c['source_file'],
     'category':    c['category'],
     'text':        c['clause_text']}
    for doc in corpus_docs
    for c in doc['clauses']
    if len(c['clause_text']) >= 40
]

clause_texts = [r['text'] for r in clause_records]
print(f'Total clauses available for embedding: {len(clause_texts)}')
print(f'Example clause_id : {clause_records[0]["clause_id"]}')
print(f'Example text       : {clause_texts[0][:100]}...')

In [ ]:
# ── Sample queries for retrieval testing ─────────────────────────────────────
QUERIES = [
    'What fees does the merchant pay?',
    'What are the termination conditions?',
    'Who owns intellectual property rights?',
    'What are the confidentiality obligations?',
    'How are disputes resolved under the agreement?',
    'What happens if a party cannot perform due to external events?',
]

# ── Embedding models to benchmark ────────────────────────────────────────────
EMBEDDING_MODELS = [
    'all-MiniLM-L6-v2',     # fast, 80 MB
    'all-mpnet-base-v2',    # accurate, 420 MB
    'BAAI/bge-small-en',    # efficient, 130 MB
]

def embed_and_retrieve(model_name, clause_texts, queries, top_k=3, verbose=True):
    """Embed all clauses + queries, return top-k results per query."""
    model = SentenceTransformer(model_name)
    clause_embs = model.encode(clause_texts, show_progress_bar=False)

    results = {}
    for query in queries:
        q_emb = model.encode([query])
        sims  = cosine_similarity(q_emb, clause_embs)[0]
        top_i = np.argsort(sims)[::-1][:top_k]
        results[query] = [
            {'rank': r+1, 'score': round(float(sims[idx]),4),
             'clause_id': clause_records[idx]['clause_id'],
             'category':  clause_records[idx]['category'],
             'text':      clause_texts[idx][:120]}
            for r, idx in enumerate(top_i)
        ]
        if verbose:
            print(f'  Q: {query}')
            for hit in results[query]:
                print(f'    [{hit["rank"]}] ({hit["score"]:.3f}) [{hit["category"]:25s}] {hit["clause_id"]:45s} {hit["text"][:60]}...')
            print()
    return results

all_retrieval_results = {}
for model_name in EMBEDDING_MODELS:
    print(f'\n{"="*70}')
    print(f'  Embedding model: {model_name}')
    print(f'{"="*70}')
    all_retrieval_results[model_name] = embed_and_retrieve(
        model_name, clause_texts, QUERIES, top_k=3
    )

In [ ]:
# ── Retrieval score table — top-1 category alignment ─────────────────────────
# For each query we check whether the top-1 retrieved clause category
# matches what we'd semantically expect.

EXPECTED_CATEGORIES = {
    'What fees does the merchant pay?':                            'Payment',
    'What are the termination conditions?':                        'Term and Termination',
    'Who owns intellectual property rights?':                      'Intellectual Property',
    'What are the confidentiality obligations?':                   'Confidentiality',
    'How are disputes resolved under the agreement?':              'Governing Law',
    'What happens if a party cannot perform due to external events?':'Force Majeure',
}

rows = []
for model_name, q_results in all_retrieval_results.items():
    correct = 0
    for query, hits in q_results.items():
        top_cat  = hits[0]['category'] if hits else 'None'
        exp_cat  = EXPECTED_CATEGORIES.get(query, '')
        matched  = int(top_cat == exp_cat)
        correct += matched
        rows.append({
            'Model':             model_name,
            'Query':             query[:55],
            'Top-1 Category':    top_cat,
            'Expected Category': exp_cat,
            'Match':             '✓' if matched else '✗',
        })

retrieval_df = pd.DataFrame(rows)
print('=== Retrieval Category Alignment per Query ===')
display(retrieval_df)

# Summary score per model
score_summary = (
    retrieval_df.groupby('Model')['Match']
    .apply(lambda s: f"{(s=='✓').sum()}/{len(s)}  ({100*(s=='✓').mean():.0f}%)")
    .reset_index()
    .rename(columns={'Match': 'Retrieval Score (top-1 category accuracy)'})
)
print('\n=== Embedding Model Retrieval Score ===')
display(score_summary)

## Export All Results

In [ ]:
# Evaluation CSV
df[['clause_id','source_file','clause_text',
    'week2_category','regex_label','ground_truth']].to_csv(
    Path(OUTPUT_DIR)/'regex_baseline_results.csv', index=False)

# Per-class metrics
per_class_df.to_csv(Path(OUTPUT_DIR)/'per_class_metrics.csv')

# Aggregate metrics
json.dump(metrics, open(Path(OUTPUT_DIR)/'aggregate_metrics.json','w'), indent=4)

# Retrieval results
retrieval_df.to_csv(Path(OUTPUT_DIR)/'retrieval_results.csv', index=False)
score_summary.to_csv(Path(OUTPUT_DIR)/'embedding_model_scores.csv', index=False)

best  = per_class_df['F1'].idxmax()
worst = per_class_df[per_class_df['Support']>0]['F1'].idxmin()
best_emb = score_summary.iloc[0]['Model'] if not score_summary.empty else 'N/A'

print('\n' + '='*62)
print('  COMBINED PIPELINE — FINAL SUMMARY')
print('='*62)
print(f'  PDFs processed            : {len(pdf_files)}')
print(f'  Total clauses extracted   : {len(all_clauses_flat)}')
print(f'  Clauses evaluated (>=40ch): {len(df)}')
print(f'  Taxonomy categories       : {len(LABELS)}')
print()
print('  — Regex Baseline Metrics —')
for k, v in metrics.items():
    print(f'    {k:22s}: {v:.4f}')
print()
print(f'  Best classified category  : {best} (F1={per_class_df.loc[best,"F1"]:.4f})')
print(f'  Worst classified category : {worst} (F1={per_class_df.loc[worst,"F1"]:.4f})')
print(f'  Best embedding model      : {best_emb}')
print('='*62)
print('\nOutput files:')
outputs = [
    'corpus_clauses.json', 'ground_truth_labels.json',
    'regex_baseline_results.csv', 'per_class_metrics.csv',
    'aggregate_metrics.json', 'baseline_evaluation.png',
    'retrieval_results.csv', 'embedding_model_scores.csv',
]
for fn in outputs:
    ok = '✓' if (Path(OUTPUT_DIR)/fn).exists() else '–'
    print(f'  [{ok}] {fn}')